# S07 — Exercises: Object-Oriented Programming

**Python for Applied Physics** | Master in Applied Physics

| Symbol | Difficulty |
|--------|------------|
| 🟢 | Accessible |
| 🟡 | Intermediate |
| 🔴 | Challenging |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import NamedTuple

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print("Setup complete")

---
## Exercise 1 🟢 — `GaussianBeam` class

Build a class representing a Gaussian beam propagating along the z-axis.

**Tasks:**
1. Define `GaussianBeam(wavelength, w0, power)` with validated `@property` attributes (`wavelength > 0`, `w0 > 0`, `power >= 0`).
2. Add read-only properties: `rayleigh_range`, `frequency`, `peak_intensity` (at waist).
3. Add methods: `w(z)` → beam radius at position z; `intensity(r, z)` → 2D intensity $I(r,z)$.
4. Add `__repr__` showing wavelength in nm, w0 in µm, power in mW.
5. Verify: `beam.intensity(0, 0) == beam.peak_intensity`, `beam.w(beam.rayleigh_range) ≈ w0 * √2`.

In [ ]:
class GaussianBeam:
    """
    A Gaussian beam propagating along the z-axis.

    Parameters
    ----------
    wavelength : float  — central wavelength (m)
    w0         : float  — beam waist radius 1/e² (m)
    power      : float  — total power (W)
    """

    def __init__(self, wavelength: float, w0: float, power: float) -> None:
        # YOUR CODE HERE
        pass

    # Properties with validation
    # YOUR CODE HERE

    # Read-only computed properties
    @property
    def rayleigh_range(self) -> float:
        """z_R = π w₀² / λ (m)."""
        # YOUR CODE HERE
        pass

    @property
    def peak_intensity(self) -> float:
        """I₀ = 2P / (π w₀²) (W/m²)."""
        # YOUR CODE HERE
        pass

    @property
    def frequency(self) -> float:
        """ν₀ = c / λ (Hz)."""
        # YOUR CODE HERE
        pass

    def w(self, z: float) -> float:
        """Beam radius at position z: w(z) = w₀ √(1 + (z/zR)²)."""
        # YOUR CODE HERE
        pass

    def intensity(self, r: np.ndarray, z: float) -> np.ndarray:
        """I(r, z) = I₀ (w₀/w(z))² exp(-2r²/w(z)²)."""
        # YOUR CODE HERE
        pass

    def __repr__(self) -> str:
        # YOUR CODE HERE
        pass


# --- Tests ---
beam = GaussianBeam(wavelength=800e-9, w0=500e-6, power=1.0)
print(repr(beam))

# Assertions
assert beam.rayleigh_range > 0
assert np.isclose(beam.w(0), beam.w0)
assert np.isclose(beam.w(beam.rayleigh_range), beam.w0 * np.sqrt(2), rtol=1e-6)
assert np.isclose(beam.intensity(np.array([0.0]), 0)[0], beam.peak_intensity, rtol=1e-6)

# Validation
try:
    GaussianBeam(wavelength=-1e-9, w0=500e-6, power=1.0)
    assert False, "Should have raised ValueError"
except ValueError:
    pass

print(f"zR             = {beam.rayleigh_range*100:.2f} cm")
print(f"peak intensity = {beam.peak_intensity/1e4:.2f} W/cm²")
print(f"frequency      = {beam.frequency/1e12:.2f} THz")
print("All assertions passed ✓")

---
## Exercise 2 🟢 — Property validation

Extend `GaussianBeam` to support $M^2 > 1$ (real beam quality).

**Tasks:**
1. Add an `M2` parameter (default `1.0`) with validation `M2 >= 1.0`.
2. Update `rayleigh_range`: $z_R = \pi w_0^2 / (M^2 \lambda)$.
3. Add a `divergence` read-only property: $\theta = M^2 \lambda / (\pi w_0)$ (half-angle, rad).
4. Verify that setting `beam.M2 = 0.5` raises a `ValueError`.
5. Show that for `M2=1` the divergence satisfies $\theta \cdot w_0 = \lambda / \pi$.

In [ ]:
class GaussianBeam:
    """Gaussian beam with M² beam quality factor."""

    c = 3e8

    def __init__(self, wavelength: float, w0: float, power: float,
                 M2: float = 1.0) -> None:
        self.wavelength = wavelength
        self.w0         = w0
        self.power      = power
        self.M2         = M2

    # YOUR CODE HERE — @property / @setter for wavelength, w0, power, M2

    @property
    def rayleigh_range(self) -> float:
        # YOUR CODE HERE — include M2
        pass

    @property
    def divergence(self) -> float:
        """Far-field half-angle divergence θ = M² λ / (π w₀) (rad)."""
        # YOUR CODE HERE
        pass

    @property
    def peak_intensity(self) -> float:
        return 2 * self.power / (np.pi * self.w0**2)

    def w(self, z: float) -> float:
        return self.w0 * np.sqrt(1 + (z / self.rayleigh_range)**2)

    def intensity(self, r: np.ndarray, z: float) -> np.ndarray:
        wz = self.w(z)
        return self.peak_intensity * (self.w0/wz)**2 * np.exp(-2*r**2/wz**2)

    def __repr__(self) -> str:
        return (f"GaussianBeam(λ={self.wavelength*1e9:.0f} nm, "
                f"w₀={self.w0*1e6:.0f} µm, P={self.power*1e3:.1f} mW, "
                f"M²={self.M2:.2f})")


# Tests
beam1 = GaussianBeam(800e-9, 500e-6, 1.0, M2=1.0)
beam2 = GaussianBeam(800e-9, 500e-6, 1.0, M2=1.5)

assert beam1.rayleigh_range > beam2.rayleigh_range, "Higher M² → shorter Rayleigh range"
assert np.isclose(beam1.divergence * beam1.w0,
                  beam1.wavelength / np.pi, rtol=1e-6), "θ · w₀ = λ/π for M²=1"

try:
    beam1.M2 = 0.5
    assert False, "Should raise ValueError"
except ValueError as e:
    print(f"Caught: {e}")

print(beam1)
print(beam2)
print(f"zR  (M²=1.0): {beam1.rayleigh_range*100:.2f} cm")
print(f"zR  (M²=1.5): {beam2.rayleigh_range*100:.2f} cm")
print(f"θ   (M²=1.0): {beam1.divergence*1e3:.3f} mrad")
print(f"θ   (M²=1.5): {beam2.divergence*1e3:.3f} mrad")
print("All assertions passed ✓")

---
## Exercise 3 🟡 — `OpticalElement` hierarchy

Build the `OpticalElement` hierarchy from the lecture and use it to simulate a beam expander.

**Tasks:**
1. Implement `OpticalElement` (abstract), `FreeSpace(d)`, `ThinLens(f)`, `Mirror(R)` as in the lecture.
2. Implement `__matmul__` on `OpticalElement` so elements can be composed with `@`.
3. Build a **5× beam expander**: `FreeSpace(f1) @ ThinLens(f1) @ FreeSpace(f1+f2) @ ThinLens(f2) @ FreeSpace(f2)` with $f_1=50\,\text{mm}$, $f_2=250\,\text{mm}$.
4. Propagate a collimated ray $[w_0, 0]$ ($w_0=1\,\text{mm}$) and verify the output beam radius is $\approx 5\,\text{mm}$.
5. Verify `isinstance(Mirror(0.3), OpticalElement) == True`.

In [ ]:
class OpticalElement:
    """Abstract base for ABCD optical elements."""

    def matrix(self) -> np.ndarray:
        raise NotImplementedError

    def propagate(self, ray: np.ndarray) -> np.ndarray:
        return self.matrix() @ ray

    def __matmul__(self, other):
        # YOUR CODE HERE
        pass

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}()"


class _MatrixElement(OpticalElement):
    def __init__(self, M): self._M = M
    def matrix(self): return self._M


class FreeSpace(OpticalElement):
    # YOUR CODE HERE
    pass


class ThinLens(OpticalElement):
    # YOUR CODE HERE
    pass


class Mirror(OpticalElement):
    # YOUR CODE HERE
    pass


# 3. 5× beam expander
f1, f2 = 50e-3, 250e-3
expander = # YOUR CODE HERE — compose with @

# 4. Propagate
w0_in  = 1e-3
ray_in = np.array([w0_in, 0.0])
ray_out = expander.propagate(ray_in)

# --- Assertions ---
assert isinstance(Mirror(0.3), OpticalElement)
assert isinstance(expander, OpticalElement)
assert np.isclose(np.abs(ray_out[0]), f2/f1 * w0_in, rtol=0.01), \
    f"Output ray height {ray_out[0]*1e3:.2f} mm ≠ expected {f2/f1*w0_in*1e3:.1f} mm"
assert np.isclose(ray_out[1], 0.0, atol=1e-6), \
    f"Output angle should be ≈0, got {ray_out[1]*1e3:.3f} mrad"

print(f"Input  : y={ray_in[0]*1e3:.1f} mm, θ={ray_in[1]*1e3:.2f} mrad")
print(f"Output : y={ray_out[0]*1e3:.2f} mm, θ={ray_out[1]*1e6:.2f} µrad")
print(f"Magnification: {abs(ray_out[0]/ray_in[0]):.2f}×")
print("All assertions passed ✓")

---
## Exercise 4 🟡 — `PulseSequence` container

Build a `PulseSequence` that acts as a full Python sequence.

**Tasks:**
1. Implement `PulseSequence` with `__len__`, `__getitem__`, `__iter__`, `__contains__`, `__add__` (concat two sequences or append a pulse).
2. Add `total_energy()`, `sort_by_energy()` (returns new sorted sequence), `filter_by_wavelength(lambda_min, lambda_max)` (returns new filtered sequence).
3. Verify that `for p in seq` works and that `p in seq` returns `True` for a contained pulse.
4. Build a sequence of 5 pulses spanning 515–1550 nm, sort by energy descending, and filter to keep only NIR ($\lambda > 800\,\text{nm}$).

In [ ]:
class GaussianPulse:
    """Minimal GaussianPulse for this exercise."""
    c = 3e8

    def __init__(self, tau, wavelength, energy):
        if tau <= 0: raise ValueError("tau must be positive")
        if wavelength <= 0: raise ValueError("wavelength must be positive")
        if energy < 0: raise ValueError("energy must be non-negative")
        self.tau = tau; self.wavelength = wavelength; self.energy = energy

    def __repr__(self):
        return (f"GaussianPulse(τ={self.tau*1e15:.0f} fs, "
                f"λ={self.wavelength*1e9:.0f} nm, E={self.energy*1e6:.1f} µJ)")

    def __eq__(self, other):
        if not isinstance(other, GaussianPulse): return NotImplemented
        return (np.isclose(self.tau, other.tau) and
                np.isclose(self.wavelength, other.wavelength) and
                np.isclose(self.energy, other.energy))

    def __lt__(self, other):
        return self.energy < other.energy


class PulseSequence:
    """An ordered collection of GaussianPulse objects."""

    def __init__(self, pulses=None):
        self._pulses = list(pulses) if pulses is not None else []

    def __len__(self):
        # YOUR CODE HERE
        pass

    def __getitem__(self, idx):
        # YOUR CODE HERE
        pass

    def __iter__(self):
        # YOUR CODE HERE
        pass

    def __contains__(self, item):
        # YOUR CODE HERE
        pass

    def __add__(self, other):
        # YOUR CODE HERE
        pass

    def __repr__(self):
        return f"PulseSequence({len(self)} pulses)"

    def total_energy(self) -> float:
        # YOUR CODE HERE
        pass

    def sort_by_energy(self, descending: bool = False) -> 'PulseSequence':
        """Return a new sequence sorted by energy."""
        # YOUR CODE HERE
        pass

    def filter_by_wavelength(
        self, lambda_min: float, lambda_max: float
    ) -> 'PulseSequence':
        """Return a new sequence keeping only pulses in [lambda_min, lambda_max]."""
        # YOUR CODE HERE
        pass


# 4. Build, sort, filter
pulses = [
    GaussianPulse(100e-15, 515e-9,  10e-6),
    GaussianPulse(150e-15, 800e-9,  50e-6),
    GaussianPulse(200e-15, 1030e-9, 80e-6),
    GaussianPulse(300e-15, 1310e-9, 35e-6),
    GaussianPulse(500e-15, 1550e-9, 20e-6),
]
seq = PulseSequence(pulses)

sorted_seq  = seq.sort_by_energy(descending=True)
nir_seq     = seq.filter_by_wavelength(800e-9, 2000e-9)

# --- Assertions ---
assert len(seq) == 5
assert pulses[0] in seq
assert len(sorted_seq) == 5
assert sorted_seq[0].energy >= sorted_seq[1].energy   # descending
assert all(p.wavelength >= 800e-9 for p in nir_seq)
assert np.isclose(seq.total_energy(), sum(p.energy for p in pulses))

# Iteration test
count = sum(1 for _ in seq)
assert count == 5

# Concatenation
extra = GaussianPulse(100e-15, 700e-9, 5e-6)
seq2  = seq + extra
assert len(seq2) == 6

print(f"Sequence          : {seq}")
print(f"Sorted by energy  : {[p.energy*1e6 for p in sorted_seq]} µJ")
print(f"NIR only          : {nir_seq}")
print(f"Total energy      : {seq.total_energy()*1e6:.1f} µJ")
print("All assertions passed ✓")

---
## Exercise 5 🟡 — `BeamParameters` dataclass

**Tasks:**
1. Define a `@dataclass(frozen=True)` called `BeamParameters` with fields: `wavelength` (m), `w0` (m), `M2=1.0`, `z=0.0`. Validate in `__post_init__`.
2. Add `@property` methods: `rayleigh_range`, `w_at_z`, `peak_intensity(power)` (as a regular method, not property, since it needs power).
3. Define a second dataclass `ShotRecord(NamedTuple)` with fields `shot_id: int`, `energy_uJ: float`, `duration_fs: float`, `beam: BeamParameters`.
4. Build a list of 5 `ShotRecord`s with varying energies and the same beam, then find the record with maximum energy.
5. Verify that `BeamParameters` instances are hashable and can be used as dict keys.

In [ ]:
from dataclasses import dataclass
from typing import NamedTuple

@dataclass(frozen=True)
class BeamParameters:
    """
    Immutable Gaussian beam parameters.
    """
    wavelength : float
    w0         : float
    M2         : float = 1.0
    z          : float = 0.0

    def __post_init__(self):
        # YOUR CODE HERE — validate wavelength > 0, w0 > 0, M2 >= 1
        pass

    @property
    def rayleigh_range(self) -> float:
        # YOUR CODE HERE
        pass

    @property
    def w_at_z(self) -> float:
        # YOUR CODE HERE
        pass

    def peak_intensity(self, power: float) -> float:
        """Peak intensity at waist I₀ = 2P / (π w₀²)."""
        # YOUR CODE HERE
        pass


class ShotRecord(NamedTuple):
    """Record of a single laser shot."""
    shot_id    : int
    energy_uJ  : float
    duration_fs: float
    beam       : BeamParameters


# 4. Build records
beam_params = BeamParameters(wavelength=800e-9, w0=400e-6, M2=1.1)

rng     = np.random.default_rng(9)
records = [
    ShotRecord(
        shot_id     = i+1,
        energy_uJ   = float(rng.normal(50, 2)),
        duration_fs = float(rng.normal(100, 3)),
        beam        = beam_params,
    )
    for i in range(5)
]

best = # YOUR CODE HERE — record with max energy_uJ

# 5. Hashability
beam_dict = {beam_params: 'main beam'}

# --- Assertions ---
assert beam_params.rayleigh_range > 0
assert np.isclose(beam_params.w_at_z, beam_params.w0)   # z=0
assert beam_params.peak_intensity(1.0) > 0
assert hash(beam_params) is not None
assert beam_dict[beam_params] == 'main beam'

# frozen: cannot modify
try:
    beam_params.w0 = 1e-3
    assert False
except Exception:
    pass

# Validation
try:
    BeamParameters(wavelength=800e-9, w0=-1e-6)
    assert False
except ValueError:
    pass

# Unpacking NamedTuple
sid, e, dur, b = records[0]
assert isinstance(b, BeamParameters)

print(beam_params)
print(f"zR = {beam_params.rayleigh_range*100:.2f} cm")
print(f"\nRecords:")
for r in records:
    print(f"  Shot {r.shot_id}: {r.energy_uJ:.2f} µJ, {r.duration_fs:.1f} fs")
print(f"\nMax energy: Shot {best.shot_id} @ {best.energy_uJ:.2f} µJ")
print("All assertions passed ✓")

---
## Exercise 6 🔴 — `LaserSystem` by composition

Build a `LaserSystem` that **contains** a pulse source and a list of optical elements. Demonstrate why composition is better than inheritance here.

**Tasks:**
1. Define `LaserSystem(source: GaussianPulse, elements: list[OpticalElement])` — use composition, not inheritance.
2. Add `propagate_ray(ray)` — propagate through all elements in order.
3. Add `add_element(element)` and `remove_last()` methods.
4. Add `output_beam_radius(w_in)` — propagate a marginal ray `[w_in, 0]` and return `|y_out|`.
5. Add `__repr__` listing source and all elements.
6. Build a system with your 800 nm pulse + 3 optical elements, compute output beam radius, and verify it differs from input.

In [ ]:
class LaserSystem:
    """
    A laser system: a pulse source + a sequence of optical elements.

    Uses composition: LaserSystem HAS-A GaussianPulse, not IS-A GaussianPulse.
    """

    def __init__(
        self,
        source: 'GaussianPulse',
        elements: list = None,
    ) -> None:
        # YOUR CODE HERE
        pass

    def propagate_ray(self, ray: np.ndarray) -> np.ndarray:
        """Propagate ray [y, θ] through all elements in order."""
        # YOUR CODE HERE
        pass

    def add_element(self, element: 'OpticalElement') -> None:
        """Append an optical element to the system."""
        # YOUR CODE HERE
        pass

    def remove_last(self) -> 'OpticalElement':
        """Remove and return the last optical element."""
        # YOUR CODE HERE
        pass

    def output_beam_radius(self, w_in: float) -> float:
        """Return output beam radius for a collimated input beam of radius w_in."""
        ray = np.array([w_in, 0.0])
        ray_out = self.propagate_ray(ray)
        return abs(ray_out[0])

    def __repr__(self) -> str:
        # YOUR CODE HERE — list source and all elements
        pass


# Use the classes from above exercises
source   = GaussianPulse(tau=100e-15, wavelength=800e-9, energy=50e-6)
sys      = LaserSystem(source, [
    FreeSpace(0.1),
    ThinLens(0.1),
    FreeSpace(0.2),
])

w_in  = 2e-3   # 2 mm input beam
w_out = sys.output_beam_radius(w_in)

# --- Assertions ---
assert isinstance(sys.source, GaussianPulse)
assert len(sys.elements) == 3
assert w_out != w_in, "Output beam radius should differ from input"

# add_element / remove_last
sys.add_element(FreeSpace(0.05))
assert len(sys.elements) == 4
removed = sys.remove_last()
assert len(sys.elements) == 3
assert isinstance(removed, FreeSpace)

print(sys)
print(f"\nInput  beam radius: {w_in*1e3:.2f} mm")
print(f"Output beam radius: {w_out*1e3:.2f} mm")
print("All assertions passed ✓")

---
## Exercise 7 🔴 — Full pulse characterisation pipeline

Integrate the OOP classes with the numerical tools from S04–S06 into a pipeline object.

**Tasks:**
1. Define a `PulseAnalyser` class that takes a `GaussianPulse` and a time array `t`.
2. Add a `run()` method that computes and stores: intensity `I_t`, spectrum `(freq, S)`, FWHM in time and frequency, TBP.
3. Add a `plot()` method that produces a two-panel figure (time + frequency) using the stored results.
4. Add a `save(path)` method that saves all results to HDF5 using `h5py`, including TBP and FWHM as attributes.
5. Test the full pipeline on a `GaussianPulse` ($\tau = 80\,\text{fs}$) and a `ChirpedPulse` ($\tau = 80\,\text{fs}$, GDD = $1000\,\text{fs}^2$). Print the TBP of both — the chirped one should be larger.

In [ ]:
import h5py, tempfile, os


class GaussianPulse:
    """Complete GaussianPulse from lecture (copy here for self-containment)."""
    c = 3e8

    def __init__(self, tau, wavelength, energy):
        if tau <= 0: raise ValueError("tau must be positive")
        if wavelength <= 0: raise ValueError("wavelength must be positive")
        if energy < 0: raise ValueError("energy must be non-negative")
        self._tau = tau; self._wavelength = wavelength; self._energy = energy

    @property
    def tau(self): return self._tau
    @property
    def wavelength(self): return self._wavelength
    @property
    def energy(self): return self._energy
    @property
    def fwhm(self): return 2 * np.sqrt(np.log(2)) * self.tau
    @property
    def bandwidth(self): return np.sqrt(np.log(2)) / (np.pi * self.tau)

    def intensity(self, t): return np.exp(-t**2 / self.tau**2)

    def spectrum(self, t):
        N = len(t); dt = t[1]-t[0]
        E_f = np.fft.fftshift(np.fft.fft(np.sqrt(self.intensity(t))))
        freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
        return freq, np.abs(E_f)**2

    def __repr__(self):
        return (f"GaussianPulse(τ={self.tau*1e15:.1f} fs, "
                f"λ={self.wavelength*1e9:.0f} nm, E={self.energy*1e6:.1f} µJ)")


class ChirpedPulse(GaussianPulse):
    def __init__(self, tau, wavelength, energy, gdd):
        super().__init__(tau, wavelength, energy)
        self._gdd = float(gdd)

    @property
    def gdd(self): return self._gdd

    @property
    def tau_chirped(self):
        return self.tau * np.sqrt(1 + (self.gdd / self.tau**2)**2)

    def intensity(self, t):
        return np.exp(-t**2 / self.tau_chirped**2)

    def spectrum(self, t):
        N = len(t); dt = t[1]-t[0]
        freq = np.fft.fftshift(np.fft.fftfreq(N, d=dt))
        ω = 2*np.pi*freq
        E_TL = np.sqrt(np.exp(-t**2/self.tau**2))
        E_f  = np.fft.fftshift(np.fft.fft(E_TL)) * np.exp(0.5j*self.gdd*ω**2)
        return freq, np.abs(E_f)**2

    def __repr__(self):
        return (f"ChirpedPulse(τ={self.tau*1e15:.1f} fs, "
                f"λ={self.wavelength*1e9:.0f} nm, GDD={self.gdd*1e30:.0f} fs²)")


# -----------------------------------------------------------------------
# PulseAnalyser
# -----------------------------------------------------------------------

class PulseAnalyser:
    """
    Characterise a GaussianPulse (or subclass) numerically.

    Usage
    -----
    analyser = PulseAnalyser(pulse, t)
    analyser.run()
    analyser.plot()
    analyser.save('result.h5')
    """

    def __init__(self, pulse: GaussianPulse, t: np.ndarray) -> None:
        # YOUR CODE HERE
        pass

    @staticmethod
    def _fwhm(x, y):
        yn = y / y.max(); above = yn >= 0.5
        return x[above].max() - x[above].min() if above.any() else np.nan

    def run(self) -> 'PulseAnalyser':
        """Compute intensity, spectrum, FWHM, TBP. Returns self for chaining."""
        # YOUR CODE HERE
        pass

    def plot(self, title: str = '') -> None:
        """Two-panel time + frequency figure."""
        # YOUR CODE HERE
        pass

    def save(self, path: str) -> None:
        """Save results to HDF5."""
        # YOUR CODE HERE
        pass


# 5. Test
N_t = 4096; dt_t = 5e-15
t_ax = (np.arange(N_t) - N_t//2) * dt_t

p_TL = GaussianPulse(80e-15, 800e-9, 50e-6)
p_ch = ChirpedPulse( 80e-15, 800e-9, 50e-6, gdd=1000e-30)

an_TL = PulseAnalyser(p_TL, t_ax).run()
an_ch = PulseAnalyser(p_ch, t_ax).run()

an_TL.plot(title='Transform-limited')
an_ch.plot(title='Chirped (GDD=1000 fs²)')

with tempfile.TemporaryDirectory() as tmp:
    an_TL.save(os.path.join(tmp, 'TL.h5'))
    an_ch.save(os.path.join(tmp, 'chirped.h5'))
    print("HDF5 files written")

# --- Assertions ---
assert hasattr(an_TL, 'tbp'), "run() must set self.tbp"
assert hasattr(an_TL, 'fwhm_t'), "run() must set self.fwhm_t"
assert an_ch.tbp > an_TL.tbp, "Chirped pulse should have higher TBP"
assert an_ch.fwhm_t > an_TL.fwhm_t, "Chirped pulse should be broader in time"

TBP_limit = 2 * np.log(2) / np.pi
print(f"\nTL     TBP = {an_TL.tbp:.4f}  (transform limit: {TBP_limit:.4f})")
print(f"Chirped TBP = {an_ch.tbp:.4f}")
print("All assertions passed ✓")

---
## Wrap-up checklist

- [ ] All 7 exercises: assertions pass
- [ ] `shared/pulse_classes.py` created with `GaussianPulse` and `ChirpedPulse`
- [ ] `shared/beam_classes.py` created with `OpticalElement`, `FreeSpace`, `ThinLens`, `Mirror`, `BeamParameters`
- [ ] Neither file modifies the existing `shared/pulses.py` or `shared/optics.py`
- [ ] `git add shared/pulse_classes.py shared/beam_classes.py`
- [ ] `git commit -m "S07: add OOP pulse and beam classes to shared/"`

**Next session:** S08 — Testing, Type Hints & Defensive Programming